# 31p — W6 PILOT (Option B refactor): distilled judge + short GRPO

**Pilot purpose**: validate direction before committing 10+ A100-hr to a full W6 run. Per the user's directive: "let's run for less time so we see we are in the correct direction, not wasting precious time and money."

## What changed vs notebook 32 (the full W6)

This pilot ships the **Option B refactor** of Component B's reward signal:

1. **Trains the distilled judge first** (cell 6, ~30 min on A100). Uses `data/reward_calibration_anchors.parquet` (210k labeled rows) → fine-tunes `cross-encoder/ms-marco-MiniLM-L-6-v2` on Gemini-aligned anchors. Output is a Hub repo `recsys2026-distilled-judge-{date}` that the GRPO reward closure loads.
2. **Reweights the reward** (already landed in `scripts/reward_fns.py`): R_retr 0.70→0.40, R_judge 0.10→0.30, R_format 0.05→0.10, R_user_prof 0.00→0.05. Useful gradient mass goes from ~0.20 to ~0.60. R_judge is now backed by a real model, not a stub.
3. **Adds intra-rollout diversity bonus** (`group_responses` kwarg in `compose_r_turn`, +0.05 cap): wires `compose_r_session.lex_div` into per-prompt training signal.
4. **Smaller scale**: 1500 sessions (10% of full), 2000 GRPO steps (vs 12k). ~1.5 A100-hr.

## Compute budget (Colab Pro: ~100 units/month)

| Step | Wallclock | Compute units |
|---|---|---|
| Distilled judge training | ~30 min A100 | ~6 |
| Retrieval pre-compute (1500 turns) | ~5 min A100 | ~1 |
| GRPO pilot (2000 steps × G=4) | ~1.5 hr A100 | ~20 |
| Format + reward delta eval (50 rows × 2 models) | ~10 min A100 | ~2 |
| **Total** | **~2.3 hr A100** | **~30 units** |

Leaves ~70 units/month for Blind-A submission inference + iteration.

## Pilot gate (decide whether to run the full W6)

PASS if both:
- format compliance ≥ 90% (slightly relaxed from full-run 95% since pilot has fewer steps)
- Δ R_turn vs B1 (W4 merged) ≥ +0.015 on the 50-row eval (half the full-run +0.03 gate, scales with step count)

PASS → schedule full W6. FAIL → iterate (more steps, different LR, or fall back to W4-only).

In [ ]:
# 1) GPU check.
!nvidia-smi | head -10

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026
!git log -1 --pretty=format:'commit:  %h%nsubject: %s'

In [ ]:
# 2b) Drive + caches.
import os, shutil
from google.colab import drive
try: drive.mount('/content/drive')
except Exception:
    try: drive.flush_and_unmount()
    except Exception: pass
    drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/recsys2026-cache'
for d in [f'{DRIVE_BASE}/hf_datasets', f'{DRIVE_BASE}/experiments_cache',
          f'{DRIVE_BASE}/judge_runs', f'{DRIVE_BASE}/grpo_pilot_runs']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_BASE}/hf_datasets'
%env HF_DATASETS_CACHE={DRIVE_BASE}/hf_datasets

EXPECTED_CACHE = '/content/recsys2026/music-crs-baselines/experiments/cache'
os.makedirs(os.path.dirname(EXPECTED_CACHE), exist_ok=True)
if os.path.exists(EXPECTED_CACHE) and not os.path.islink(EXPECTED_CACHE):
    shutil.rmtree(EXPECTED_CACHE)
if not os.path.islink(EXPECTED_CACHE):
    os.symlink(f'{DRIVE_BASE}/experiments_cache', EXPECTED_CACHE)

In [ ]:
# 3) HF auth.
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
from huggingface_hub import whoami
print(f'✓ HF auth ok — {whoami(token=HF_TOKEN)["name"]}')

In [ ]:
# 4) Resolve starting base = W5 merged > W4 merged.
import json
from pathlib import Path

def latest_gate(runs_dir):
    p = Path(runs_dir)
    if not p.exists(): return None
    cands = list(p.rglob('gate_result.json'))
    if not cands: return None
    latest = max(cands, key=lambda x: x.stat().st_mtime)
    with latest.open() as f: return json.load(f)

W5 = latest_gate(f'{DRIVE_BASE}/sdpo_runs')
W4 = latest_gate(f'{DRIVE_BASE}/kto_runs')

STARTING_MERGED = None
PRIOR_STAGE = None
B1_REPO = None
if W5 and W5.get('merged_hub_model'):
    STARTING_MERGED = W5['merged_hub_model']; PRIOR_STAGE = 'W5'
elif W4 and W4.get('merged_hub_model'):
    STARTING_MERGED = W4['merged_hub_model']; PRIOR_STAGE = 'W4'
B1_REPO = W4.get('merged_hub_model') if W4 else None

if STARTING_MERGED is None:
    raise SystemExit('❌ no merged base from W4/W5. Run notebook 30 (W4) first.')
print(f'starting from {PRIOR_STAGE}: {STARTING_MERGED}')
print(f'B1 baseline (W4 merged) for reward delta: {B1_REPO}')

In [ ]:
# 5) Install deps + pytest pre-flight (incl. new W7 tests).
!pip install -q --upgrade transformers datasets 'pandas<3.0' tqdm omegaconf
!pip install -q --upgrade 'trl>=0.12.0' 'peft>=0.13.0' 'torchao>=0.16.0' trackio accelerate
!python -c 'import torch, transformers, trl, peft; print("torch", torch.__version__, "trl", trl.__version__, "peft", peft.__version__)'

!cd /content/recsys2026 && python -m pytest \
    tests/test_reward_fns.py \
    tests/test_build_grpo_dataset.py \
    tests/test_train_distilled_judge.py \
    tests/test_build_train_plus_dev.py \
    -q

In [ ]:
# 6) Train the distilled judge (~30 min on A100).
#
# This is the highest-leverage step in the Option B refactor: replaces
# r_judge_stub (always returns 0) with a cross-encoder that approximates
# Gemini's judgment. Without this, R_judge contributes 0 gradient.
from datetime import date
from pathlib import Path

JUDGE_HUB_REPO = ff'{HF_USERNAME}/recsys2026-distilled-judge-{date.today().isoformat()}'
JUDGE_OUT_DIR = '/content/recsys2026/distilled_judge_run'
JUDGE_DATA_DIR = '/content/recsys2026/data/distilled_judge'

# Step 6a — prepare JSONL pairs (CPU; ~1 min).
!cd /content/recsys2026 && python scripts/train_distilled_judge.py \
    --mode prepare \
    --anchors data/reward_calibration_anchors.parquet \
    --out-dir {JUDGE_DATA_DIR} \
    --val-frac 0.1 --seed 42

# Step 6b — train cross-encoder (~30 min A100).
!cd /content/recsys2026 && python scripts/train_distilled_judge.py \
    --mode train \
    --data-dir {JUDGE_DATA_DIR} \
    --base-model cross-encoder/ms-marco-MiniLM-L-6-v2 \
    --output-dir {JUDGE_OUT_DIR} \
    --hub-repo {JUDGE_HUB_REPO} \
    --epochs 2 --batch-size 64 --lr 2e-5 --max-length 512

print(f'✓ distilled judge at https://huggingface.co/{JUDGE_HUB_REPO}')

In [ ]:
# 7) Build the pilot GRPO parquet (small scale).
#
# 1500 sessions instead of 15000 → ~10% data. Runs the same pipeline:
# build_reward_dataset → augment_envelope → retrieval pre-compute →
# build_grpo_dataset, but with a smaller --n-sessions.
#
# Reuses cells from notebook 32 cell 6/7 — see that for the full retrieval
# pre-compute logic. We only retrieve for the 1500-session subset to keep
# the pre-compute step under 10 min.
import json, sys, re
from pathlib import Path
import pandas as pd
from tqdm import tqdm

N_SESSIONS_PILOT = 1500
REWARD = '/content/recsys2026/data/reward_train_pilot.parquet'
ENV_PATH = '/content/recsys2026/data/reward_train_envelope_pilot.parquet'
RETRIEVAL_OUT = '/content/recsys2026/data/trl/grpo_retrieval_pilot.parquet'
GRPO_OUT = '/content/recsys2026/data/trl/grpo_pilot.parquet'

if not Path(REWARD).exists():
    !cd /content/recsys2026 && python scripts/build_reward_dataset.py \
        --hf-split train --n-sessions {N_SESSIONS_PILOT} \
        --split-val 0.1 --out {REWARD}
if not Path(ENV_PATH).exists():
    !cd /content/recsys2026 && python scripts/augment_envelope.py --in {REWARD} --out {ENV_PATH}

# Retrieval pre-compute — same code as notebook 32 cell 7 but smaller input.
if Path(RETRIEVAL_OUT).exists():
    print(f'reusing existing {RETRIEVAL_OUT}')
else:
    sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
    import torch
    from mcrs.retrieval_modules import load_retrieval_module
    from mcrs.rerankers.pro_rank import ProRankReranker
    from mcrs.query_rewriters.cmqr import CMQR_REWRITER
    from mcrs.query_rewriters.state_tracker import StateTracker
    from mcrs.lm_modules import load_lm_module
    from mcrs.db_item import MusicCatalogDB

    ITEM_DB = 'talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
    DATASET = 'talkpl-ai/TalkPlayData-Challenge-Dataset'
    SPLITS = ['all_tracks']
    CORPUS = ['track_name', 'artist_name', 'album_name']
    CACHE = '/content/recsys2026/music-crs-baselines/experiments/cache'
    LM_TYPE = 'meta-llama/Llama-3.2-1B-Instruct'
    PROMPTS = '/content/recsys2026/music-crs-baselines/mcrs/system_prompts'

    lm = load_lm_module(lm_type=LM_TYPE, device='cuda', attn_implementation='sdpa', dtype=torch.bfloat16, use_vllm=False)
    retrieval = load_retrieval_module('wrrf_bm25_dense_lyrics_v1', ITEM_DB, SPLITS, CORPUS, CACHE)
    state_tracker = StateTracker(lm=lm, prompt_path=f'{PROMPTS}/state_extraction.txt', cache_dir=CACHE, max_new_tokens=96)
    cmqr = CMQR_REWRITER(lm=lm, inner_retriever=retrieval, prompt_path=f'{PROMPTS}/cmqr_rewrites.txt',
                         cache_dir=CACHE, n_rewrites=4, topk_per_rewrite=50, rrf_k=60, max_new_tokens=96)
    reranker = ProRankReranker(item_db_name=ITEM_DB, track_split_types=SPLITS, corpus_types=CORPUS, cache_dir=CACHE, with_rationales=True)
    item_db = MusicCatalogDB(dataset_name=ITEM_DB, split_types=SPLITS, corpus_types=CORPUS)
    valid_catalog = set(item_db.metadata_dict.keys())

    env_df = pd.read_parquet(ENV_PATH).query('label == 1').drop_duplicates(['session_id', 'turn_number'])
    print(f'unique POS turns: {len(env_df):,}')

    from datasets import load_dataset
    raw = load_dataset(DATASET, split='train')
    sid2turn_to_gold = {}
    sid2turn_to_user_query = {}
    sid2turn_to_history = {}
    for sess in raw:
        sid = sess['session_id']
        msgs = sorted(sess['conversations'], key=lambda m: (int(m['turn_number']), m['role']))
        for msg in msgs:
            if msg['role'] == 'music':
                sid2turn_to_gold[(sid, int(msg['turn_number']))] = str(msg['content'])
        for tn in range(1, 9):
            prior = [m for m in msgs if int(m['turn_number']) < tn]
            history = '\n'.join(f"{m['role']}: {m['content']}" for m in prior)
            user_msg = next((m for m in msgs if int(m['turn_number']) == tn and m['role'] == 'user'), None)
            if user_msg is None: continue
            sid2turn_to_user_query[(sid, tn)] = str(user_msg['content'])
            sid2turn_to_history[(sid, tn)] = history

    rows_out = []
    BATCH = 16
    rows_buffer = []
    for i, row in enumerate(tqdm(env_df.itertuples(index=False), total=len(env_df), desc='retrieve')):
        rows_buffer.append((row.session_id, int(row.turn_number)))
        if len(rows_buffer) < BATCH and i + 1 < len(env_df): continue
        sids = [s for s, _ in rows_buffer]; tns = [t for _, t in rows_buffer]
        queries = [sid2turn_to_user_query.get((s, t), '') for s, t in rows_buffer]
        histories = [sid2turn_to_history.get((s, t), '') for s, t in rows_buffer]
        states = []
        for s, t, q, h in zip(sids, tns, queries, histories):
            try: states.append(state_tracker.extract(s, t, q, h))
            except Exception: states.append(None)
        cmqr.set_batch_context(session_ids=sids, turn_numbers=tns, extracted_states=states)
        try: top100 = cmqr.batch_text_to_item_retrieval(queries, topk=100, user_ids=[None]*len(queries))
        except TypeError: top100 = cmqr.batch_text_to_item_retrieval(queries, topk=100)
        top20 = reranker.rerank(queries, top100, topk=20)
        for s, t, q, ids20, pool in zip(sids, tns, queries, top20, top100):
            seen, kept = set(), []
            for tid in ids20:
                if tid in seen or tid not in valid_catalog: continue
                kept.append(tid); seen.add(tid)
            if len(kept) < 20:
                for tid in pool:
                    if len(kept) >= 20: break
                    if tid in seen or tid not in valid_catalog: continue
                    kept.append(tid); seen.add(tid)
            kept = kept[:20]
            try: rats = reranker.generate_rationales(q, kept)
            except Exception: rats = [''] * len(kept)
            top1_tid = kept[0] if kept else ''
            top1_meta = item_db.metadata_dict.get(top1_tid, {}) if top1_tid else {}
            tn_get = lambda f: (top1_meta.get(f) or [''])
            tn1 = (tn_get('track_name')[0] if isinstance(tn_get('track_name'), list) else str(tn_get('track_name'))) or ''
            an1 = (tn_get('artist_name')[0] if isinstance(tn_get('artist_name'), list) else str(tn_get('artist_name'))) or ''
            rows_out.append({
                'session_id': s, 'turn_number': t,
                'gold_track_id': sid2turn_to_gold.get((s, t), ''),
                'predicted_track_ids': kept, 'top1_track_name': tn1,
                'top1_artist_name': an1, 'reranker_rationales': rats,
            })
        rows_buffer.clear()

    out_df = pd.DataFrame(rows_out).drop_duplicates(['session_id', 'turn_number'], keep='last')
    Path(RETRIEVAL_OUT).parent.mkdir(parents=True, exist_ok=True)
    out_df.to_parquet(RETRIEVAL_OUT, index=False)
    print(f'✓ retrieval cache: {len(out_df):,} rows')

# Build the GRPO parquet via the conversational-prompt path.
SYSTEM_PROMPT_TXT = '/content/recsys2026/data/trl/grpo_system_prompt.txt'
PROMPTS_DIR = '/content/recsys2026/music-crs-baselines/mcrs/system_prompts'
with open(f'{PROMPTS_DIR}/roleplay.txt', encoding='utf-8') as f: rp = f.read()
with open(f'{PROMPTS_DIR}/response_generation_cot_user_state.txt', encoding='utf-8') as f: cp = f.read()
Path(SYSTEM_PROMPT_TXT).parent.mkdir(parents=True, exist_ok=True)
with open(SYSTEM_PROMPT_TXT, 'w', encoding='utf-8') as f: f.write(rp + '\n\n' + cp)

if not Path(GRPO_OUT).exists():
    !cd /content/recsys2026 && python scripts/build_grpo_dataset.py \
        --envelope {ENV_PATH} --retrieval {RETRIEVAL_OUT} \
        --system-prompt-path {SYSTEM_PROMPT_TXT} --out {GRPO_OUT}

import pandas as pd
d = pd.read_parquet(GRPO_OUT)
print(f'\npilot GRPO: {len(d):,} rows')

In [ ]:
# 8) Schema + 90/10 split.
from datasets import Dataset
import numpy as np

def _norm(p):
    if isinstance(p, np.ndarray): return [dict(m) for m in p]
    if isinstance(p, list) and p and isinstance(p[0], np.ndarray): return [dict(m) for m in p]
    return p
d['prompt'] = d['prompt'].apply(_norm)
ds = Dataset.from_pandas(d, preserve_index=False)
split = ds.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split['train'], split['test']
print(f'train: {len(train_ds):,}  eval: {len(eval_ds):,}')

In [ ]:
# 9) Trackio init + GRPO pilot training (Option B reward closure).
#
# This is where the Option B refactor lands at training:
#   - reward_main calls compose_r_turn(judge_score=judge.score(ctx, response),
#     group_responses=...) with the new weights.
#   - judge is a DistilledJudge instance loading the cross-encoder we trained
#     in cell 6.
#   - reward_format is a separate fn (kept from W6 review P1-2).
from datetime import date
import trackio
from transformers import TrainerCallback

RUN_NAME = f'b3-grpo-pilot-{date.today().isoformat()}'
TRACKIO_OK = True
try:
    trackio.init(project='recsys2026', name=RUN_NAME, group='b-stage',
                 config={'pilot': True, 'judge': JUDGE_HUB_REPO, 'starting': STARTING_MERGED,
                         'n_sessions': N_SESSIONS_PILOT, 'max_steps': 2000})
    print(f'✓ Trackio: {RUN_NAME}')
except Exception as e:
    TRACKIO_OK = False
    print(f'⚠️  trackio failed: {e!r}')

class TrackioCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not TRACKIO_OK or not logs: return
        try: trackio.log({k: float(v) for k, v in logs.items() if isinstance(v, (int, float))})
        except Exception: pass

import torch, gc, json, sys
from peft import LoraConfig
from trl import GRPOTrainer, GRPOConfig
sys.path.insert(0, '/content/recsys2026/scripts')
from reward_fns import compose_r_turn, r_format, DistilledJudge

# Instantiate the distilled judge — lazy-loads on first .score() call.
JUDGE = DistilledJudge(checkpoint=JUDGE_HUB_REPO)
# Deep-review P1-4: warmup before trainer init.
JUDGE.warmup()

HUB_REPO = ff'{HF_USERNAME}/recsys2026-{RUN_NAME}'
OUTPUT_DIR = f'/content/recsys2026/training_runs/{RUN_NAME}'

peft_config = LoraConfig(r=32, lora_alpha=32, lora_dropout=0.05,
                         bias='none', task_type='CAUSAL_LM', target_modules='all-linear')

def _completion_text(c):
    if isinstance(c, list) and c:
        last = c[-1]
        if isinstance(last, dict): return str(last.get('content', ''))
        return str(last)
    return str(c)


def _user_content_from_prompt(prompt):
    """Extract the user-role content string for the judge's `context` arg."""
    if isinstance(prompt, list):
        for msg in prompt:
            if isinstance(msg, dict) and msg.get('role') == 'user':
                return str(msg.get('content', ''))
        return ''
    return str(prompt)


def reward_main(prompts, completions, **kwargs):
    """Composite Option B reward (no format gate, format split below)."""
    # Group completion indices by (session_id, turn_number) for the
    # group_responses (intra-rollout diversity) bonus.
    from collections import defaultdict
    groups = defaultdict(list)
    sids = kwargs['session_id']; tns = kwargs['turn_number']
    for i in range(len(completions)):
        groups[(sids[i], tns[i])].append(i)

    # Pre-compute user-role contexts for the judge in one batch.
    contexts = [_user_content_from_prompt(p) for p in prompts]
    completion_texts = [_completion_text(c) for c in completions]
    # Batch the judge calls — the cross-encoder forward pass is the bulk
    # of the per-step cost. score_batch returns 0.0 per row if no checkpoint.
    judge_scores = JUDGE.score_batch(contexts, completion_texts, batch_size=16)

    scores = []
    for i, completion in enumerate(completions):
        # Peer responses for diversity bonus.
        peer_indices = groups[(sids[i], tns[i])]
        peer_responses = [completion_texts[j] for j in peer_indices]
        comps = compose_r_turn(
            predicted_track_ids=list(kwargs['predicted_track_ids'][i]),
            gold_track_id=kwargs['gold_track_id'][i],
            response_text=completion_texts[i],
            valid_catalog=None,
            top1_meta=json.loads(kwargs['top1_meta_json'][i]),
            user_state=json.loads(kwargs['user_state_json'][i]),
            user_profile=json.loads(kwargs['user_profile_json'][i]) if 'user_profile_json' in kwargs else None,  # gap-analysis Step 3: user_profile piped
            history_text=kwargs['history_text'][i],
            judge_score=judge_scores[i],
            include_format=False,
            group_responses=peer_responses,
        )
        scores.append(comps['r_turn'])
    return scores


def reward_format(prompts, completions, **kwargs):
    return [r_format(_completion_text(c)) for c in completions]


config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    push_to_hub=True, hub_model_id=HUB_REPO, hub_strategy='every_save', hub_private_repo=True,
    num_generations=4, scale_rewards=False, max_completion_length=320,
    temperature=0.9, beta=0.04, reward_weights=[0.95, 0.05],
    max_steps=2_000,                       # PILOT: was 12_000 in full
    per_device_train_batch_size=1, gradient_accumulation_steps=4,
    learning_rate=5e-6, lr_scheduler_type='cosine', warmup_ratio=0.05,
    bf16=True, gradient_checkpointing=True,
    eval_strategy='steps', eval_steps=400, per_device_eval_batch_size=2,
    save_strategy='steps', save_steps=200, save_total_limit=2,
    logging_steps=20, report_to='none',
)

callbacks = [TrackioCallback()] if TRACKIO_OK else []
trainer = GRPOTrainer(
    model=STARTING_MERGED, args=config,
    train_dataset=train_ds, eval_dataset=eval_ds,
    reward_funcs=[reward_main, reward_format],
    peft_config=peft_config, callbacks=callbacks,
)
print(f'🚀 PILOT GRPO (~1.5 A100-hr expected)…')
trainer.train()
print('✓ pilot training complete')

In [ ]:
# 10) Push adapter + in-place merge.
import gc, torch
from transformers import AutoTokenizer

trainer.push_to_hub()
trainer.optimizer = None; trainer.lr_scheduler = None
gc.collect(); torch.cuda.empty_cache()

fully_merged = trainer.model.merge_and_unload()
tok = AutoTokenizer.from_pretrained(STARTING_MERGED)
MERGED_REPO = ff'{HF_USERNAME}/recsys2026-{RUN_NAME}-merged'
fully_merged.push_to_hub(MERGED_REPO, private=True,
                         commit_message=f'PILOT W6 Option B merged on {STARTING_MERGED}')
tok.push_to_hub(MERGED_REPO, private=True)
print(f'✓ pilot merged: https://huggingface.co/{MERGED_REPO}')

In [ ]:
# 11) Format compliance + reward delta vs B1 — RELAXED PILOT GATE.
#
# Pilot gate: format ≥ 90% (not 95%); Δ R_turn ≥ +0.015 (not +0.03).
import json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import sys
sys.path.insert(0, '/content/recsys2026/scripts')
from reward_fns import r_format, ENVELOPE, compose_r_turn

w6_tok = AutoTokenizer.from_pretrained(MERGED_REPO)
w6_model = AutoModelForCausalLM.from_pretrained(MERGED_REPO, torch_dtype=torch.bfloat16, device_map='cuda').eval()
b1_model = None
if B1_REPO:
    b1_model = AutoModelForCausalLM.from_pretrained(B1_REPO, torch_dtype=torch.bfloat16, device_map='cuda').eval()

def gen(model, tok, conv):
    formatted = tok.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
    enc = tok(formatted, return_tensors='pt', truncation=True, max_length=2048).to('cuda')
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=320, do_sample=False,
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, enc['input_ids'].shape[1]:], skip_special_tokens=True)

n_eval = min(50, len(eval_ds))
n_strict = n_loose = 0
samples = []
w6_rs, b1_rs = [], []
for i in range(n_eval):
    ex = eval_ds[i]
    out = gen(w6_model, w6_tok, ex['prompt'])
    if r_format(out) == 1.0: n_strict += 1
    if ENVELOPE.search(out): n_loose += 1
    if len(samples) < 3: samples.append(out[:400])
    w6_rs.append(compose_r_turn(
        predicted_track_ids=list(ex['predicted_track_ids']),
        gold_track_id=ex['gold_track_id'], response_text=out, valid_catalog=None,
        top1_meta=json.loads(ex['top1_meta_json']),
        user_state=json.loads(ex['user_state_json']),
        history_text=ex['history_text'], judge_score=JUDGE.score(
            ex['prompt'][1]['content'] if isinstance(ex['prompt'], list) and len(ex['prompt']) > 1 else '', out
        ),
    )['r_turn'])
    if b1_model:
        b1_out = gen(b1_model, w6_tok, ex['prompt'])
        b1_rs.append(compose_r_turn(
            predicted_track_ids=list(ex['predicted_track_ids']),
            gold_track_id=ex['gold_track_id'], response_text=b1_out, valid_catalog=None,
            top1_meta=json.loads(ex['top1_meta_json']),
            user_state=json.loads(ex['user_state_json']),
            history_text=ex['history_text'], judge_score=JUDGE.score(
                ex['prompt'][1]['content'] if isinstance(ex['prompt'], list) and len(ex['prompt']) > 1 else '', b1_out
            ),
        )['r_turn'])

format_strict = n_strict / n_eval
format_loose = n_loose / n_eval
mean_w6 = sum(w6_rs) / max(len(w6_rs), 1)
mean_b1 = sum(b1_rs) / max(len(b1_rs), 1) if b1_rs else None
delta = (mean_w6 - mean_b1) if mean_b1 is not None else None

print(f'\nPILOT format strict: {format_strict:.1%} (gate: ≥90%)')
print(f'PILOT format loose:   {format_loose:.1%}')
print(f'PILOT W6 R_turn:      {mean_w6:.4f}')
if mean_b1 is not None:
    print(f'PILOT B1 R_turn:      {mean_b1:.4f}')
    print(f'PILOT Δ vs B1:        {delta:+.4f}  (gate: ≥+0.015)')

print('\nSamples:')
for i, s in enumerate(samples, 1): print(f'\n--- {i} ---\n{s}')

gate_format = format_strict >= 0.90
gate_reward = (delta is not None and delta >= 0.015)

print('\n' + '=' * 60)
if gate_format and gate_reward:
    print(f'PILOT PASS — schedule full W6 (notebook 32, 12k steps).')
elif gate_format:
    print(f'PILOT PARTIAL — format OK, reward delta below threshold. Tune & re-pilot.')
else:
    print(f'PILOT FAIL — format regressed. Revisit reward weights or fall back to W4.')
print('=' * 60)

# Persist gate result.
import os
result = {
    'stage': 'B3-pilot',
    'run_name': RUN_NAME,
    'starting_base': STARTING_MERGED,
    'judge': JUDGE_HUB_REPO,
    'merged_hub_model': MERGED_REPO,
    'format_compliance_strict': format_strict,
    'mean_w6_r_turn': mean_w6,
    'mean_b1_r_turn': mean_b1,
    'delta_r_turn_vs_b1': delta,
    'gate_format_passed': gate_format,
    'gate_reward_passed': gate_reward,
    'pilot_passed': gate_format and gate_reward,
    'sample_outputs': samples,
}
out_path = f'{DRIVE_BASE}/grpo_pilot_runs/{RUN_NAME}/gate_result.json'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, 'w', encoding='utf-8') as f: json.dump(result, f, ensure_ascii=False, indent=2)
print(f'\ngate_result → {out_path}')
if TRACKIO_OK: trackio.finish()

## Pilot decision tree

**PILOT PASS** (format ≥ 90% AND Δ R_turn ≥ +0.015):
- Schedule full W6 via `colab/32_train_responder_grpo.ipynb` (12k steps, 10 A100-hr).
- The full run can reuse this notebook's distilled judge — set `JUDGE_HUB_REPO` in cell 11 of notebook 32.
- Optional: also run `colab/41_run_blindset_A.ipynb` against the pilot merged model for cheap Gemini-judge calibration.

**PILOT PARTIAL** (format OK, reward delta short):
- Likely cause: 2k steps wasn't enough convergence. Try doubling to 4k and re-piloting. ~3 A100-hr.
- Or: nudge `learning_rate` 5e-6 → 1e-5; raise `kl_beta` if format started slipping near the end.

**PILOT FAIL** (format regressed):
- Revert: ship W4-merged or W5-merged directly to Blind-A as a no-RL baseline; skip W6 entirely.
- Likely cause: reward weights too aggressive. Test A/B by reverting to v1 weights (R_retr=0.70) for one run, OR halve `W_JUDGE` (the new piece) to see if the judge is mis-calibrated.